In [1]:
pip install beautifulsoup4 fake-useragent pandas requests nltk transformers datasets tf-keras --upgrade tensorflow

Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.



In [2]:
from datasets import load_dataset
from transformers import pipeline
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import nltk
nltk.download('vader_lexicon')
nltk.download('words')

from nltk.sentiment.vader import SentimentIntensityAnalyzer as SIA

C:\Users\mraeg\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\mraeg\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package words to
[nltk_data]     C:\Users\mraeg\AppData\Roaming\nltk_data...
[nltk_data]   Package words is already up-to-date!


In [ ]:
# Your ScraperAPI key
API_KEY = "ee7788f0f60ba626a14ffd2088e844c1"  # Replace with your actual ScraperAPI key

# Function to get HTML content using ScraperAPI with JavaScript rendering enabled
def get_html(url, retries=3):
    params = {
        "api_key": API_KEY,
        "url": url,
        "country_code": "us",
        "render": "true"  # Enable JavaScript rendering to load expanded content
    }
    for attempt in range(retries):
        response = requests.get("http://api.scraperapi.com", params=params)
        if response.status_code == 200:
            return response.text
        else:
            print(f"Attempt {attempt + 1} failed with status: {response.status_code}. Retrying...")
            time.sleep(random.uniform(1, 3))  # Wait before retrying
    print("Failed to retrieve data after multiple attempts.")
    return None

# Base URL for Amazon product reviews with pageNumber placeholder
asin = "B099N1LC4R"  # Replace with your desired ASIN
base_url = f"https://www.amazon.com/product-reviews/{asin}/?pageNumber={{page}}&reviewerType=all_reviews&filterByStar"
# List to store all scraped reviews
all_reviews = []
review_texts = []

# Define maximum number of pages to scrape
max_pages = 10  # Set this to the number of pages you want to scrape
analyzer = SIA()
# Loop through each page
for page in range(1, max_pages + 1):
    print(f"Scraping page {page}")
    url = base_url.format(page=page)
    html = get_html(url)
    if not html:
        print(f"Stopping due to empty response on page {page}")
        break  # Stop if the page didn't load correctly
    
    # Parse the HTML content with BeautifulSoup
    soup = BeautifulSoup(html, 'html.parser')
    
    # Check if there are reviews on this page
    reviews_on_page = soup.find_all('div', {'data-hook': 'review'})
    if not reviews_on_page:
        print("No more reviews found, stopping pagination.")
        break  # Stop if there are no more reviews on this page

    # Extract details from each review
    for review in reviews_on_page:
        try:
            # Review text
            review_text = review.find('span', {'data-hook': 'review-body'}).get_text(strip=True)

            # Review title
            review_title = review.find('a', {'data-hook': 'review-title'}).get_text(strip=True) if review.find('a', {'data-hook': 'review-title'}) else None

            # Review date
            review_date = review.find('span', {'data-hook': 'review-date'}).get_text(strip=True)

            # Star rating
            review_rating = review.find('i', {'data-hook': 'review-star-rating'}).get_text(strip=True) if review.find('i', {'data-hook': 'review-star-rating'}) else None
            
            # Verified Purchase status
            verified = review.find('span', text="Verified Purchase")
            is_verified = "Yes" if verified else "No"
            
            # Helpful votes (if available)
            helpful_votes = review.find('span', {'data-hook': 'helpful-vote-statement'})
            if helpful_votes:
                helpful_text = helpful_votes.get_text(strip=True)
                # Extract numeric value from text (e.g., "2 people found this helpful" -> 2)
                helpful_count = int(helpful_text.split()[0]) if helpful_text.split()[0].isdigit() else 0
            else:
                helpful_count = 0

            # Reviewer's name
            reviewer_name = review.find('span', {'class': 'a-profile-name'}).get_text(strip=True) if review.find('span', {'class': 'a-profile-name'}) else None

            # Review Images (if any)
            images = review.find_all('img', {'data-hook': 'review-image-tile'})
            image_links = [img['src'] for img in images] if images else None
            
            # Append the review data to the list
            all_reviews.append({
                "Reviewer Name": reviewer_name,
                "Review Title": review_title,
                "ReviewDescription": review_text,
                "Date": review_date,
                "Rating": review_rating,
                "Verified Purchase": is_verified,
                "Helpful Votes": helpful_count,
                "Image Links": image_links
            })

            review_texts.append({
                "Reviews": review_text
            })
        except AttributeError:
            # Skip reviews with missing fields
            continue

    # Check for "Next" button to continue pagination
    next_button = soup.find('li', {'class': 'a-last'})
    if next_button and next_button.find('a'):
        page += 1
    else:
        print("No more pages available, stopping pagination.")
        break

    # Random delay between pages to mimic human behavior
    time.sleep(random.uniform(2, 5))

print(f"Total reviews scraped: {len(all_reviews)}")


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\mraeg\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package words to
[nltk_data]     C:\Users\mraeg\AppData\Roaming\nltk_data...
[nltk_data]   Package words is already up-to-date!
[nltk_data] Downloading package maxent_ne_chunker to
[nltk_data]     C:\Users\mraeg\AppData\Roaming\nltk_data...
[nltk_data]   Package maxent_ne_chunker is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mraeg\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Error loading average_perception_tagger: Package
[nltk_data]     'average_perception_tagger' not found in index


Scraping page 1


C:\Users\mraeg\AppData\Local\Temp\ipykernel_20744\2910190939.py:79: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  verified = review.find('span', text="Verified Purchase")


Scraping page 2
Scraping page 3
Scraping page 4
Scraping page 5
Scraping page 6
Scraping page 7
Scraping page 8
Scraping page 9
Scraping page 10
No more pages available, stopping pagination.
Total reviews scraped: 100
Reviews saved to '/Users/mraeg/Downloads/amazon_reviews_moisturizer_cetaphil_mattg.csv'


In [4]:
df_import = pd.read_csv('ostomy pouch.csv')

#Reads all entries under 'Review' column into a list
reviews_column = df_import.loc[:, 'Review'].values.tolist()
reviews_column = [str(r) for r in reviews_column]

print(reviews_column)
#Creates DataFrame of review_column list
df_reviews = pd.DataFrame(reviews_column, columns=['Reviews'])

#Sentiment Analysis
#analyzer is a Sentiment analysis object we call methods from
analyzer = SIA()
#Creates 'compound' and 'sentiment' columns in dataframe
df_reviews['compound'] = [analyzer.polarity_scores(x)['compound'] for x in df_reviews['Reviews']]
df_reviews['sentiment'] = 'neutral'
#sets all columns with specific score ranges to equal postive or negative
df_reviews.loc[df_reviews.compound > 0.05, 'sentiment'] = 'positive'
df_reviews.loc[df_reviews.compound < -0.05, 'sentiment'] = 'negative'

#resets index of both dataframes
df_reviews.index = df_reviews.index.astype(int)
df_import.index = df_import.index.astype(int)

print(df_reviews)

#Emotional analysis
#model: uses 'j-hartmann/emotion-english-distilroberta-base' analysis model in a "text-classification" format
model = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base")

#Creates dataframe using 'j-hartmann/emotion-english-distilroberta-base' to show 'label' and 'score'
all_emotions = model(reviews_column, truncation = True)

#Creates another DataFrame with one column for 'Reviews'
df_emotions = pd.DataFrame(reviews_column, columns=['Reviews'])

#Reads from emotions model DataFrame and puts it into main DataFrame we will output
df_emotions["emotion_label"] = [d["label"] for d in all_emotions]
df_emotions["emotion_score"] = [d["score"] for d in all_emotions]

print(df_emotions)

#Joins the main dataframe with sentiment analysis dataframe
df_import= df_import.join(df_reviews, lsuffix='_df_import', rsuffix='_df_reviews')
df_import= df_import.join(df_emotions, lsuffix='_df_import', rsuffix='_df_emotions')
df_import = df_import.reset_index()
print(df_reviews)
df_import.drop(columns=['Reviews_df_emotions'], inplace= True, axis = 1)
df_import.drop(columns=['Reviews_df_import'], inplace= True, axis = 1)
#df_reviews['Review Title'] = df_reviews['Review Title'].str[18:].astype(str)
#df_reviews['Rating'] = df_reviews['Rating'].str[:4].astype(str)
#df_reviews['Date'] = df_reviews['Date'].str[32:].astype(str)


# Save all reviews to a DataFrame and write to a CSV file
save_path = "/Users/mraeg/Downloads/ostomy_pouch_test.csv"  # Adjust the path as needed
df_import.to_csv(save_path, index=False)
print(f"Reviews saved to '{save_path}'")

["The glue that holds the velcro drain closed is not even strong enough to hold the bag closed. The velcro holds but the glue on the bag let's go of the velcro and just opens in the middle of the night. This is not a one off either. Every pouch out of this lot has done it so far.", "I've been using this product for a few month now...it works well for 2-3days.*2 cons: occasionally leakage & sometimes the built in air filter hits my stoma, which hurts & has caused me to bleed once. However the affordability has been a blessing.", "I have a subscription for these every month for my nephew, and he hasn't had any problems with them. The only problem I do have is how often they are delayed. My nephew needs these asap. NOT the next day or the day after that!", 'Leaks all the time', 'These bag are ok for home use, they do stick better than any I have tried. My problem is that EVERY SINGLE BRAND claims to be odor proof and NONE OF THEM ARE. This one is no exception.If anyone can recommend a bra